# 2.4 Data Engineering Essentials - Tutorial

**Primary outcome:** Build an end-to-end ETL pipeline from raw synthetic e-commerce data into a clean SQLite warehouse, with data quality checks throughout.

**Time estimate:** ~60–90 minutes

---

## Learning Objectives

By the end of this tutorial you will be able to:

- Explain what ETL (Extract, Transform, Load) means and why it matters
- Generate and extract data from multiple synthetic sources
- Clean, standardise, and enrich raw data using pandas
- Store cleaned data in a SQLite warehouse and query it back
- Write automated data quality checks that produce a quality report
- Wrap the full pipeline into a single reusable `run_etl()` function

---

## Prerequisites

- Python basics (functions, loops, dictionaries)
- pandas fundamentals (DataFrames, filtering, groupby)
- Completion of modules 2.1–2.3 is helpful but not required

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

# Reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"sqlite3 : {sqlite3.sqlite_version}")

---

## 1. What is Data Engineering?

**Data engineering** is the practice of designing and building systems that collect, store, and process raw data so that data analysts and scientists can use it reliably.

The most common pattern is the **ETL pipeline**:

| Phase | What happens |
|-------|--------------|
| **Extract** | Pull raw data from one or more sources (databases, APIs, files, streams) |
| **Transform** | Clean, validate, reshape, and enrich the data |
| **Load** | Write the clean data into a target store (data warehouse, database, data lake) |

Modern variants include **ELT** (load first, transform inside the warehouse) and **streaming pipelines**, but the ETL mental model remains foundational.

### Why does it matter?

> "Data scientists spend 80% of their time cleaning data" — Forbes (2016).
> Data engineering automates that work so it happens *before* the analyst sees the data.

### Key responsibilities of a data engineer

- Build and maintain **data pipelines**
- Ensure **data quality** and reliability
- Manage **data storage** (databases, warehouses, lakes)
- Enable **data integration** across systems
- Schedule and **orchestrate** pipeline runs

In [ ]:
# ASCII diagram: the ETL pipeline we will build in this notebook
diagram = """
+------------------+      EXTRACT      +-------------------+
|  Raw Sources     |  ------------->   |  Raw DataFrames   |
|  orders          |                   |  (dirty data)     |
|  customers       |                   +-------------------+
|  products        |                            |
+------------------+                   TRANSFORM |
                                                  v
                                    +-------------------+
                                    |  Clean DataFrames |
                                    |  (validated,      |
                                    |   enriched,       |
                                    |   aggregated)     |
                                    +-------------------+
                                                  |
                                         LOAD     |
                                                  v
                                    +-------------------+
                                    |  SQLite Warehouse |
                                    |  warehouse.db     |
                                    +-------------------+
                                                  |
                                        VALIDATE  |
                                                  v
                                    +-------------------+
                                    |  Quality Report   |
                                    +-------------------+
"""
print(diagram)

---

## 2. Extract: Loading Data from Multiple Sources

In a real pipeline, data arrives from APIs, CSV files, or operational databases.
Here we **generate synthetic e-commerce data** so the lesson focuses on the pipeline mechanics rather than any external dependency.

We will create three tables:

| Table | Rows | Key columns |
|-------|------|-------------|
| `orders` | 1 000 | order_id, customer_id, product_id, quantity, amount, date, status |
| `customers` | 200 | customer_id, name, email, city, country, signup_date |
| `products` | 50 | product_id, name, category, price |

We intentionally inject **dirty data**: nulls, duplicates, invalid dates, and negative amounts — so we have real problems to fix in the Transform phase.

In [ ]:
def generate_customers(n=200):
    """Generate a synthetic customers table."""
    np.random.seed(42)
    first_names = ["Alice", "Bob", "Carol", "David", "Eva", "Frank", "Grace", "Hiro", "Iris", "Jack"]
    last_names  = ["Smith", "Jones", "Lee", "Patel", "Nguyen", "Brown", "Wilson", "Kim", "Müller", "Santos"]
    cities      = ["Dubai", "Riyadh", "Abu Dhabi", "Jeddah", "Manama", "Doha", "Kuwait City", "Muscat"]
    countries   = ["UAE", "Saudi Arabia", "Bahrain", "Qatar", "Kuwait", "Oman"]

    base_date = datetime(2021, 1, 1)
    signup_dates = [base_date + timedelta(days=int(d)) for d in np.random.randint(0, 730, n)]

    df = pd.DataFrame({
        "customer_id" : range(1, n + 1),
        "name"        : [f"{np.random.choice(first_names)} {np.random.choice(last_names)}" for _ in range(n)],
        "email"       : [f"user{i}@example.com" for i in range(1, n + 1)],
        "city"        : np.random.choice(cities, n),
        "country"     : np.random.choice(countries, n),
        "signup_date" : signup_dates,
    })

    # --- Inject dirty data ---
    # Nulls in city
    null_idx = np.random.choice(df.index, size=10, replace=False)
    df.loc[null_idx, "city"] = np.nan
    # Duplicate rows
    df = pd.concat([df, df.iloc[:5]], ignore_index=True)
    # Mixed-case names
    df.loc[df.index[:20], "name"] = df.loc[df.index[:20], "name"].str.upper()

    return df


customers_raw = generate_customers(200)
print(f"customers_raw shape : {customers_raw.shape}")
print(customers_raw.head(3).to_string())

In [ ]:
def generate_products(n=50):
    """Generate a synthetic products table."""
    np.random.seed(42)
    categories = ["Electronics", "Clothing", "Books", "Home & Garden", "Sports"]
    cat_prices  = {"Electronics": (100, 1500), "Clothing": (20, 200),
                   "Books": (10, 80), "Home & Garden": (15, 300), "Sports": (25, 500)}

    cats   = np.random.choice(categories, n)
    prices = np.array([round(np.random.uniform(*cat_prices[c]), 2) for c in cats])

    df = pd.DataFrame({
        "product_id" : range(1, n + 1),
        "name"       : [f"Product_{i}" for i in range(1, n + 1)],
        "category"   : cats,
        "price"      : prices,
    })

    # --- Inject dirty data ---
    # Null prices
    df.loc[np.random.choice(df.index, size=3, replace=False), "price"] = np.nan
    # Inconsistent category casing
    df.loc[df.index[:5], "category"] = df.loc[df.index[:5], "category"].str.lower()

    return df


products_raw = generate_products(50)
print(f"products_raw shape : {products_raw.shape}")
print(products_raw.head(3).to_string())

In [ ]:
def generate_orders(n=1000, n_customers=200, n_products=50):
    """Generate a synthetic orders table."""
    np.random.seed(42)
    statuses   = ["completed", "pending", "cancelled", "refunded"]
    base_date  = datetime(2022, 1, 1)
    order_dates = [base_date + timedelta(days=int(d)) for d in np.random.randint(0, 365, n)]

    df = pd.DataFrame({
        "order_id"    : range(1001, 1001 + n),
        "customer_id" : np.random.randint(1, n_customers + 1, n),
        "product_id"  : np.random.randint(1, n_products + 1, n),
        "quantity"    : np.random.randint(1, 6, n),
        "amount"      : np.round(np.random.uniform(10, 2000, n), 2),
        "date"        : order_dates,
        "status"      : np.random.choice(statuses, n, p=[0.7, 0.15, 0.1, 0.05]),
    })

    # --- Inject dirty data ---
    # Null amounts
    df.loc[np.random.choice(df.index, size=30, replace=False), "amount"] = np.nan
    # Negative amounts (data entry errors)
    df.loc[np.random.choice(df.index, size=15, replace=False), "amount"] = -99.99
    # Duplicate orders
    df = pd.concat([df, df.iloc[:10]], ignore_index=True)
    # Bad date strings (simulate a broken export)
    # Cast to object dtype first so a string value can be assigned into the column
    df["date"] = df["date"].astype(object)
    df.loc[df.index[:5], "date"] = "NOT_A_DATE"
    # Mixed-case status
    df.loc[df.index[50:70], "status"] = df.loc[df.index[50:70], "status"].str.upper()

    return df


orders_raw = generate_orders(1000)
print(f"orders_raw shape : {orders_raw.shape}")
print(orders_raw.head(3).to_string())

print("\n✓ Extracted 1000 orders, 200 customers, 50 products")

In [ ]:
# Inspect the raw data for problems
print("=== orders_raw null counts ===")
print(orders_raw.isnull().sum())

print("\n=== orders_raw duplicate rows ===")
print(f"Duplicates: {orders_raw.duplicated().sum()}")

print("\n=== Negative amounts ===")
print(f"Count: {(orders_raw['amount'] < 0).sum()}")

print("\n=== Status values (mixed case) ===")
print(orders_raw['status'].value_counts().to_string())

---

## 3. Transform: Cleaning and Standardising

The raw data has several problems we need to fix before it is safe to analyse:

| Problem | Fix |
|---------|-----|
| Null values | `fillna()` with sensible defaults, or `dropna()` for required fields |
| Duplicate rows | `drop_duplicates()` |
| Wrong data types | `pd.to_datetime()`, `astype()` |
| Negative amounts | Filter rows where `amount < 0` |
| Inconsistent text | `str.lower()`, `str.strip()` |

In [ ]:
def clean_orders(df):
    """Clean the raw orders DataFrame."""
    df = df.copy()

    # 1. Remove exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    print(f"  Removed {before - len(df)} duplicate rows")

    # 2. Parse dates — coerce bad strings to NaT
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # 3. Drop rows where date could not be parsed (required field)
    before = len(df)
    df = df.dropna(subset=["date"])
    print(f"  Dropped {before - len(df)} rows with unparseable dates")

    # 4. Fill null amounts with the column median
    median_amount = df["amount"].median()
    null_count = df["amount"].isnull().sum()
    df["amount"] = df["amount"].fillna(median_amount)
    print(f"  Filled {null_count} null amounts with median ({median_amount:.2f})")

    # 5. Remove rows with negative amounts
    before = len(df)
    df = df[df["amount"] >= 0]
    print(f"  Removed {before - len(df)} rows with negative amounts")

    # 6. Normalise text columns
    df["status"] = df["status"].str.lower().str.strip()

    return df


print("Cleaning orders...")
orders_clean = clean_orders(orders_raw)
print(f"\norders: {len(orders_raw)} rows  →  {len(orders_clean)} rows after cleaning")

In [ ]:
def clean_customers(df):
    """Clean the raw customers DataFrame."""
    df = df.copy()

    # 1. Remove duplicates (keep first occurrence)
    before = len(df)
    df = df.drop_duplicates(subset=["customer_id"])
    print(f"  Removed {before - len(df)} duplicate customers")

    # 2. Fill missing city with 'Unknown'
    null_count = df["city"].isnull().sum()
    df["city"] = df["city"].fillna("Unknown")
    print(f"  Filled {null_count} missing cities with 'Unknown'")

    # 3. Normalise text: title-case names, lowercase email
    df["name"]  = df["name"].str.strip().str.title()
    df["email"] = df["email"].str.strip().str.lower()
    df["city"]  = df["city"].str.strip().str.title()

    # 4. Parse signup_date
    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

    return df


def clean_products(df):
    """Clean the raw products DataFrame."""
    df = df.copy()

    # 1. Remove duplicates
    df = df.drop_duplicates(subset=["product_id"])

    # 2. Fill null prices with category median
    null_count = df["price"].isnull().sum()
    df["price"] = df.groupby("category")["price"].transform(lambda x: x.fillna(x.median()))
    print(f"  Filled {null_count} null prices with category median")

    # 3. Standardise category casing
    df["category"] = df["category"].str.strip().str.title()
    df["name"]     = df["name"].str.strip()

    return df


print("Cleaning customers...")
customers_clean = clean_customers(customers_raw)

print("\nCleaning products...")
products_clean = clean_products(products_raw)

print(f"\ncustomers: {len(customers_raw)} → {len(customers_clean)} rows")
print(f"products : {len(products_raw)} → {len(products_clean)} rows")
print("\n✓ All three tables cleaned")

In [ ]:
# Verify: null counts after cleaning
print("Null counts after cleaning:")
print("\n--- orders ---")
print(orders_clean.isnull().sum())
print("\n--- customers ---")
print(customers_clean.isnull().sum())
print("\n--- products ---")
print(products_clean.isnull().sum())

---

## 4. Transform: Enrichment and Aggregation

Cleaning removes bad data. **Enrichment** adds value by:

- **Joining** tables to combine information
- **Deriving new columns** (e.g. `revenue`, `order_month`, `days_since_signup`)
- **Aggregating** into summary tables for reporting

These enriched/aggregated tables become the "gold layer" of your warehouse — the tables analysts actually query.

In [ ]:
def enrich_orders(orders, customers, products):
    """Join orders with customers and products, then derive new columns."""

    # Merge orders → customers (left join to keep all orders)
    df = orders.merge(
        customers[["customer_id", "name", "city", "country", "signup_date"]],
        on="customer_id",
        how="left"
    )

    # Merge → products
    df = df.merge(
        products[["product_id", "name", "category", "price"]].rename(
            columns={"name": "product_name"}
        ),
        on="product_id",
        how="left"
    )

    # Rename customer name column for clarity
    df = df.rename(columns={"name": "customer_name"})

    # Derived columns
    df["revenue"]          = (df["quantity"] * df["price"]).round(2)
    df["order_month"]      = df["date"].dt.to_period("M").astype(str)
    df["days_since_signup"] = (df["date"] - df["signup_date"]).dt.days

    return df


orders_enriched = enrich_orders(orders_clean, customers_clean, products_clean)

print(f"Enriched orders shape: {orders_enriched.shape}")
print("\nSample columns:")
print(orders_enriched[["order_id", "customer_name", "product_name",
                        "quantity", "price", "revenue", "order_month"]].head(5).to_string())

In [ ]:
# Aggregate 1: Monthly revenue
monthly_revenue = (
    orders_enriched
    .groupby("order_month", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        num_orders=("order_id", "count"),
        avg_order_value=("revenue", "mean")
    )
    .sort_values("order_month")
)
monthly_revenue["total_revenue"]     = monthly_revenue["total_revenue"].round(2)
monthly_revenue["avg_order_value"]   = monthly_revenue["avg_order_value"].round(2)

print("Monthly Revenue Summary")
print(monthly_revenue.to_string(index=False))

In [ ]:
# Aggregate 2: Top 10 customers by revenue
top_customers = (
    orders_enriched
    .groupby(["customer_id", "customer_name"], as_index=False)
    .agg(total_revenue=("revenue", "sum"), num_orders=("order_id", "count"))
    .sort_values("total_revenue", ascending=False)
    .head(10)
)
top_customers["total_revenue"] = top_customers["total_revenue"].round(2)

print("Top 10 Customers by Revenue")
print(top_customers.to_string(index=False))

In [ ]:
# Aggregate 3: Top 10 products by revenue
top_products = (
    orders_enriched
    .groupby(["product_id", "product_name", "category"], as_index=False)
    .agg(total_revenue=("revenue", "sum"), units_sold=("quantity", "sum"))
    .sort_values("total_revenue", ascending=False)
    .head(10)
)
top_products["total_revenue"] = top_products["total_revenue"].round(2)

print("Top 10 Products by Revenue")
print(top_products.to_string(index=False))
print("\n✓ Enrichment and aggregation complete")

---

## 5. Load: Storing to SQLite

The **Load** phase writes the clean, enriched data into a persistent store.

We use **SQLite** — a file-based SQL database that ships with Python's standard library. In production you would use Postgres, BigQuery, Snowflake, or Redshift, but the pattern is identical: `df.to_sql()` writes the DataFrame, and `pd.read_sql()` reads it back.

```
df.to_sql(table_name, connection, if_exists='replace', index=False)
pd.read_sql(query, connection)
```

We use `:memory:` so nothing is written to disk during this tutorial, but swapping to `"warehouse.db"` would persist the file.

In [ ]:
def load_to_warehouse(orders, customers, products,
                      orders_enriched, monthly_revenue,
                      top_customers, top_products):
    """Write all clean tables to an in-memory SQLite warehouse."""

    conn = sqlite3.connect(":memory:")

    tables = {
        "orders"          : orders,
        "customers"       : customers,
        "products"        : products,
        "orders_enriched" : orders_enriched,
        "monthly_revenue" : monthly_revenue,
        "top_customers"   : top_customers,
        "top_products"    : top_products,
    }

    for table_name, df in tables.items():
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"  Loaded '{table_name}' ({len(df)} rows)")

    return conn


print("Loading tables into SQLite warehouse...")
conn = load_to_warehouse(
    orders_clean, customers_clean, products_clean,
    orders_enriched, monthly_revenue, top_customers, top_products
)
print("\n✓ All tables loaded into warehouse.db (in-memory)")

In [ ]:
# Verify the load by querying back
# List all tables in the warehouse
table_list = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
print("Tables in warehouse:")
print(table_list.to_string(index=False))

In [ ]:
# Sample SQL query: total revenue by category
query = """
    SELECT
        category,
        COUNT(*)          AS num_orders,
        SUM(revenue)      AS total_revenue,
        AVG(revenue)      AS avg_revenue
    FROM orders_enriched
    WHERE revenue IS NOT NULL
    GROUP BY category
    ORDER BY total_revenue DESC
"""
revenue_by_category = pd.read_sql(query, conn)
revenue_by_category["total_revenue"] = revenue_by_category["total_revenue"].round(2)
revenue_by_category["avg_revenue"]   = revenue_by_category["avg_revenue"].round(2)

print("Revenue by Product Category (from warehouse)")
print(revenue_by_category.to_string(index=False))

In [ ]:
# Another query: completed orders in Q1 2022
query_q1 = """
    SELECT
        order_month,
        COUNT(*)     AS completed_orders,
        SUM(revenue) AS q1_revenue
    FROM orders_enriched
    WHERE status = 'completed'
      AND order_month IN ('2022-01', '2022-02', '2022-03')
    GROUP BY order_month
    ORDER BY order_month
"""
q1_results = pd.read_sql(query_q1, conn)
q1_results["q1_revenue"] = q1_results["q1_revenue"].round(2)

print("Completed Orders – Q1 2022")
print(q1_results.to_string(index=False))
print("\n✓ Warehouse queries verified")

---

## 6. Data Quality Checks

Production pipelines should **never silently produce wrong results**. We encode quality expectations as a `validate_pipeline()` function that:

1. Checks for nulls in key columns
2. Checks for negative amounts
3. Validates row counts are within expected ranges
4. Validates date ranges
5. Returns a structured **quality report** so you can see pass/fail at a glance

Running this on **both raw and clean data** makes the improvement quantifiable.

In [ ]:
def validate_pipeline(orders_df, customers_df, products_df, label="dataset"):
    """
    Run a suite of data quality checks.

    Returns a dict with one entry per check:
        { check_name: {"passed": bool, "detail": str} }
    """
    report = {}

    # --- Check 1: No nulls in required order columns ---
    required_order_cols = ["order_id", "customer_id", "product_id", "amount", "date"]
    null_counts = orders_df[required_order_cols].isnull().sum()
    total_nulls = int(null_counts.sum())
    report["orders_no_nulls_in_key_cols"] = {
        "passed": total_nulls == 0,
        "detail": f"{total_nulls} nulls found in {required_order_cols}"
    }

    # --- Check 2: No negative amounts ---
    neg_count = int((orders_df["amount"] < 0).sum())
    report["orders_no_negative_amounts"] = {
        "passed": neg_count == 0,
        "detail": f"{neg_count} rows with amount < 0"
    }

    # --- Check 3: Row count in expected range ---
    order_rows = len(orders_df)
    report["orders_row_count_reasonable"] = {
        "passed": 500 <= order_rows <= 2000,
        "detail": f"{order_rows} rows (expected 500–2000)"
    }

    # --- Check 4: No duplicate order IDs ---
    dup_count = int(orders_df.duplicated(subset=["order_id"]).sum())
    report["orders_no_duplicate_ids"] = {
        "passed": dup_count == 0,
        "detail": f"{dup_count} duplicate order_id values"
    }

    # --- Check 5: Date range plausible ---
    try:
        dates = pd.to_datetime(orders_df["date"], errors="coerce").dropna()
        min_date = dates.min()
        max_date = dates.max()
        date_range_ok = (min_date >= pd.Timestamp("2020-01-01")) and \
                        (max_date <= pd.Timestamp("2030-12-31"))
        report["orders_dates_in_range"] = {
            "passed": date_range_ok,
            "detail": f"Date range: {min_date.date()} → {max_date.date()}"
        }
    except Exception as e:
        report["orders_dates_in_range"] = {"passed": False, "detail": str(e)}

    # --- Check 6: All customer cities non-null ---
    city_nulls = int(customers_df["city"].isnull().sum())
    report["customers_city_not_null"] = {
        "passed": city_nulls == 0,
        "detail": f"{city_nulls} null city values"
    }

    # --- Check 7: No null product prices ---
    price_nulls = int(products_df["price"].isnull().sum())
    report["products_price_not_null"] = {
        "passed": price_nulls == 0,
        "detail": f"{price_nulls} null price values"
    }

    return report


def print_quality_report(report, label):
    passed = sum(1 for v in report.values() if v["passed"])
    total  = len(report)
    print(f"\n{'='*50}")
    print(f" Quality Report: {label}  ({passed}/{total} checks passed)")
    print(f"{'='*50}")
    for check, result in report.items():
        status = "PASS" if result["passed"] else "FAIL"
        icon   = "✓" if result["passed"] else "✗"
        print(f"  {icon} [{status}] {check}")
        print(f"         {result['detail']}")
    print()


print("Running quality checks on RAW data...")
raw_report = validate_pipeline(orders_raw, customers_raw, products_raw, label="RAW")
print_quality_report(raw_report, "RAW DATA")

In [ ]:
print("Running quality checks on CLEAN data...")
clean_report = validate_pipeline(orders_clean, customers_clean, products_clean, label="CLEAN")
print_quality_report(clean_report, "CLEAN DATA")

# Summary comparison
raw_pass   = sum(1 for v in raw_report.values()   if v["passed"])
clean_pass = sum(1 for v in clean_report.values() if v["passed"])
total      = len(raw_report)
print(f"Quality improvement: {raw_pass}/{total} → {clean_pass}/{total} checks passing")
print("✓ Data quality validation complete")

---

## 7. Mini Pipeline: Putting It All Together

A production ETL pipeline is typically wrapped in a single callable function so it can be:

- Triggered on a schedule (daily, hourly)
- Tested as a unit
- Monitored for failures
- Re-run on demand

The `run_etl()` function below calls every step in sequence and prints a summary report — exactly what a data engineer would hand off to an orchestrator like Airflow or Prefect.

In [ ]:
def run_etl():
    """
    Full ETL pipeline:
      1. Extract  – generate synthetic source data
      2. Transform – clean, enrich, aggregate
      3. Load      – write to SQLite warehouse
      4. Validate  – run quality checks

    Returns the warehouse connection and the quality report.
    """
    pipeline_start = datetime.now()
    print("="*55)
    print(" ETL PIPELINE START")
    print("="*55)

    # ── EXTRACT ─────────────────────────────────────────
    print("\n[1/4] EXTRACT")
    orders    = generate_orders(1000)
    customers = generate_customers(200)
    products  = generate_products(50)
    print(f"  ✓ Extracted {len(orders)} orders, "
          f"{len(customers)} customers, "
          f"{len(products)} products")

    # ── TRANSFORM ───────────────────────────────────────
    print("\n[2/4] TRANSFORM")
    orders_c    = clean_orders(orders)
    customers_c = clean_customers(customers)
    products_c  = clean_products(products)
    enriched    = enrich_orders(orders_c, customers_c, products_c)
    monthly_rev = (
        enriched.groupby("order_month", as_index=False)
        .agg(total_revenue=("revenue", "sum"),
             num_orders=("order_id", "count"))
        .sort_values("order_month")
    )
    top_cust = (
        enriched.groupby(["customer_id", "customer_name"], as_index=False)
        .agg(total_revenue=("revenue", "sum"))
        .sort_values("total_revenue", ascending=False)
        .head(10)
    )
    top_prod = (
        enriched.groupby(["product_id", "product_name", "category"], as_index=False)
        .agg(total_revenue=("revenue", "sum"), units_sold=("quantity", "sum"))
        .sort_values("total_revenue", ascending=False)
        .head(10)
    )
    print(f"  ✓ Transformed: {len(orders_c)} clean orders, "
          f"{len(enriched)} enriched rows, "
          f"{len(monthly_rev)} monthly periods")

    # ── LOAD ────────────────────────────────────────────
    print("\n[3/4] LOAD")
    warehouse = load_to_warehouse(
        orders_c, customers_c, products_c,
        enriched, monthly_rev, top_cust, top_prod
    )
    print("  ✓ All tables written to warehouse")

    # ── VALIDATE ────────────────────────────────────────
    print("\n[4/4] VALIDATE")
    quality_report = validate_pipeline(orders_c, customers_c, products_c)
    checks_passed  = sum(1 for v in quality_report.values() if v["passed"])
    total_checks   = len(quality_report)
    print(f"  ✓ {checks_passed}/{total_checks} quality checks passed")

    if checks_passed < total_checks:
        print("  ⚠ Some checks failed — review the quality report")

    # ── SUMMARY ─────────────────────────────────────────
    elapsed = (datetime.now() - pipeline_start).total_seconds()
    print("\n" + "="*55)
    print(" PIPELINE SUMMARY")
    print("="*55)
    print(f"  Runtime          : {elapsed:.2f}s")
    print(f"  Orders loaded    : {len(orders_c)}")
    print(f"  Customers loaded : {len(customers_c)}")
    print(f"  Products loaded  : {len(products_c)}")
    total_revenue = enriched["revenue"].sum()
    print(f"  Total revenue    : ${total_revenue:,.2f}")
    print(f"  Quality score    : {checks_passed}/{total_checks}")
    print("="*55)

    return warehouse, quality_report


warehouse_conn, final_report = run_etl()

---

## 8. Practice Exercises

Complete the three exercises below. Solutions are collapsed in the next cells — try each exercise yourself before looking.

---

### Exercise 1: Add a New Quality Check

Extend `validate_pipeline()` with a new check:
**"All order statuses are one of: completed, pending, cancelled, refunded"**

- Use `df['status'].isin([...])` to find invalid values
- The check should `PASS` when no unexpected statuses exist
- Test it on both `orders_raw` and `orders_clean`

*Hint: remember the raw data has some UPPER-CASE status values.*

In [ ]:
# Exercise 1: Your solution here


<details>
<summary><strong>Solution – Exercise 1</strong> (click to expand)</summary>

```python
def check_valid_statuses(orders_df):
    valid_statuses = {"completed", "pending", "cancelled", "refunded"}
    actual = set(orders_df["status"].dropna().unique())
    invalid = actual - valid_statuses
    return {
        "passed": len(invalid) == 0,
        "detail": f"Invalid statuses found: {invalid}" if invalid else "All statuses valid"
    }

print("RAW  :", check_valid_statuses(orders_raw))
print("CLEAN:", check_valid_statuses(orders_clean))
```

The raw data will **FAIL** (UPPER-CASE variants not in the valid set), while the clean data will **PASS** after `str.lower()` normalisation.

</details>

---

### Exercise 2: Add a New Transformation

Add a column called **`order_size`** to `orders_enriched` that categorises each order by quantity:

| quantity | order_size |
|----------|------------|
| 1 | "small" |
| 2–3 | "medium" |
| 4+ | "large" |

Then compute the **average revenue per order_size** using `groupby()`.

*Hint: use `pd.cut()` or `np.select()` to create the category column.*

In [ ]:
# Exercise 2: Your solution here


<details>
<summary><strong>Solution – Exercise 2</strong> (click to expand)</summary>

```python
conditions = [
    orders_enriched["quantity"] == 1,
    orders_enriched["quantity"].between(2, 3),
    orders_enriched["quantity"] >= 4,
]
choices = ["small", "medium", "large"]
orders_enriched["order_size"] = np.select(conditions, choices, default="unknown")

avg_revenue_by_size = (
    orders_enriched
    .groupby("order_size", as_index=False)
    .agg(avg_revenue=("revenue", "mean"), num_orders=("order_id", "count"))
    .round(2)
)
print(avg_revenue_by_size.to_string(index=False))
```

</details>

---

### Exercise 3: Query the Warehouse

Using `pd.read_sql()` against `warehouse_conn`, write a SQL query to answer:

> **"Which country generated the most revenue from completed orders?"**

Your result should show `country`, `total_revenue`, and `num_orders`, sorted by `total_revenue` descending.

*Hint: the `orders_enriched` table in the warehouse has `country` and `revenue` columns.*

In [ ]:
# Exercise 3: Your solution here


<details>
<summary><strong>Solution – Exercise 3</strong> (click to expand)</summary>

```python
query = """
    SELECT
        country,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(*)               AS num_orders
    FROM orders_enriched
    WHERE status = 'completed'
      AND revenue IS NOT NULL
    GROUP BY country
    ORDER BY total_revenue DESC
"""
result = pd.read_sql(query, warehouse_conn)
print(result.to_string(index=False))
```

</details>

---

## Summary

In this tutorial you built a complete **ETL pipeline** from scratch:

| Phase | What you did | Key tools |
|-------|-------------|----------|
| **Extract** | Generated 3 synthetic source tables with intentional dirty data | `numpy`, `pandas` |
| **Transform – Clean** | Removed duplicates, fixed nulls, parsed dates, filtered invalid rows, normalised text | `drop_duplicates`, `fillna`, `dropna`, `to_datetime`, `str.lower` |
| **Transform – Enrich** | Joined tables, derived `revenue`, `order_month`, `days_since_signup` | `merge`, column arithmetic, `dt` accessor |
| **Transform – Aggregate** | Built monthly revenue, top customers, top products summary tables | `groupby`, `agg` |
| **Load** | Wrote all tables to an in-memory SQLite warehouse | `sqlite3`, `to_sql`, `read_sql` |
| **Validate** | Automated 7 quality checks, compared raw vs clean scores | Custom `validate_pipeline()` |
| **Orchestrate** | Wrapped everything in `run_etl()` with a printed summary | Python functions |

### Key takeaways

- **Separate concerns**: keep Extract, Transform, and Load as distinct functions — it makes pipelines testable and debuggable.
- **Fail loudly**: quality checks should surface problems immediately, not let bad data silently flow downstream.
- **Automate everything**: a pipeline that requires manual steps will eventually be skipped.
- **The tools scale**: the same pandas + SQL pattern used here works on BigQuery, Snowflake, or Spark with minimal changes.

---

## Next Steps

- **Module 3 – Statistical Analysis**: now that you can build clean data pipelines, learn to extract statistical insights from the data.
- **Explore real orchestrators**: [Apache Airflow](https://airflow.apache.org/), [Prefect](https://www.prefect.io/), or [dbt](https://www.getdbt.com/) implement production-grade versions of the patterns you practised here.
- **Try a real warehouse**: replace `:memory:` with a file path (`"warehouse.db"`) to persist your SQLite database, or try [DuckDB](https://duckdb.org/) for a faster analytical engine with the same API.
- **Add scheduling**: wrap `run_etl()` in a cron job or a cloud function to run it automatically every day.